In [2]:
import math

In [3]:
# ==========================================
# STEP 1: Input sentence
# ==========================================

text = "what is the main purpose of transformer architecture"
tokens = text.split()

print("Tokens:", tokens)


Tokens: ['what', 'is', 'the', 'main', 'purpose', 'of', 'transformer', 'architecture']


In [4]:
# ==========================================
# STEP 2: Create illustrative input vectors
# ==========================================

# 8 tokens, 4 dimensions per token
X = []

for i, token in enumerate(tokens):
    X.append([
        (i + 1) * 0.1,
        (i + 1) * 0.2,
        (i + 1) * 0.3,
        (i + 1) * 0.4
    ])

In [5]:
# ==========================================
# STEP 3: Matrix utilities
# ==========================================

def transpose(A):
    return [list(row) for row in zip(*A)]


def matmul(A, B):
    BT = transpose(B)
    result = []

    for row in A:
        output_row = []

        for col in BT:
            output_row.append(
                sum(a * b for a, b in zip(row, col))
            )

        result.append(output_row)

    return result


def softmax(values):
    max_value = max(values)
    exps = [math.exp(x - max_value) for x in values]
    total = sum(exps)

    return [x / total for x in exps]


In [6]:
# ==========================================
# STEP 4: Define two attention heads
# ==========================================

# Identity projections are used for learning.
# Real models learn separate projection weights.

WQ1 = [
    [1, 0],
    [0, 1],
    [0, 0],
    [0, 0]
]

WK1 = [
    [1, 0],
    [0, 1],
    [0, 0],
    [0, 0]
]

WV1 = [
    [1, 0],
    [0, 1],
    [0, 0],
    [0, 0]
]

WQ2 = [
    [0, 0],
    [0, 0],
    [1, 0],
    [0, 1]
]

WK2 = [
    [0, 0],
    [0, 0],
    [1, 0],
    [0, 1]
]

WV2 = [
    [0, 0],
    [0, 0],
    [1, 0],
    [0, 1]
]


In [7]:
# ==========================================
# STEP 5: Scaled dot-product attention
# ==========================================

def attention(Q, K, V):
    scores = matmul(Q, transpose(K))

    d_k = len(K[0])
    scores = [
        [x / math.sqrt(d_k) for x in row]
        for row in scores
    ]

    weights = [softmax(row) for row in scores]

    output = matmul(weights, V)

    return output, weights

In [8]:
# ==========================================
# STEP 6: Multi-head attention
# ==========================================

# Head 1 projections
Q1 = matmul(X, WQ1)
K1 = matmul(X, WK1)
V1 = matmul(X, WV1)

head1, weights1 = attention(Q1, K1, V1)

# Head 2 projections
Q2 = matmul(X, WQ2)
K2 = matmul(X, WK2)
V2 = matmul(X, WV2)

head2, weights2 = attention(Q2, K2, V2)

In [9]:
# ==========================================
# STEP 7: Concatenate head outputs
# ==========================================

concatenated = []

for row1, row2 in zip(head1, head2):
    concatenated.append(row1 + row2)


In [10]:
# ==========================================
# STEP 8: Output projection
# ==========================================

# Identity output projection for demonstration.
WO = [
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 1]
]

final_output = matmul(concatenated, WO)


In [11]:
# ==========================================
# STEP 9: Display results
# ==========================================

print("\nHead 1 output:")
for token, vector in zip(tokens, head1):
    print(token, [round(x, 4) for x in vector])

print("\nHead 2 output:")
for token, vector in zip(tokens, head2):
    print(token, [round(x, 4) for x in vector])

print("\nConcatenated output shape:")
print(len(concatenated), "x", len(concatenated[0]))

print("\nFinal multi-head output:")
for token, vector in zip(tokens, final_output):
    print(token, [round(x, 4) for x in vector])


Head 1 output:
what [0.4685, 0.9371]
is [0.4869, 0.9738]
the [0.505, 1.01]
main [0.5227, 1.0454]
purpose [0.5398, 1.0796]
of [0.5563, 1.1126]
transformer [0.572, 1.1441]
architecture [0.587, 1.174]

Head 2 output:
what [1.6194, 2.1592]
is [1.8434, 2.4579]
the [2.0061, 2.6748]
main [2.1166, 2.8222]
purpose [2.1908, 2.9211]
of [2.2416, 2.9888]
transformer [2.2775, 3.0367]
architecture [2.3037, 3.0716]

Concatenated output shape:
8 x 4

Final multi-head output:
what [0.4685, 0.9371, 1.6194, 2.1592]
is [0.4869, 0.9738, 1.8434, 2.4579]
the [0.505, 1.01, 2.0061, 2.6748]
main [0.5227, 1.0454, 2.1166, 2.8222]
purpose [0.5398, 1.0796, 2.1908, 2.9211]
of [0.5563, 1.1126, 2.2416, 2.9888]
transformer [0.572, 1.1441, 2.2775, 3.0367]
architecture [0.587, 1.174, 2.3037, 3.0716]
